# <center>Scene com <br><p><center>Coverage Map(CM) e Paths</center>

## <center>Inicialização do sistema para correr o código</center>
Inicialmente para podermos correr o código devemos definir as configurações do sistema

In [ ]:
import os                              #Importa a biblioteca os
gpu_num = 0 # Use "" to use the CPU    #Aqui configuramos se va
os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Colab does currently not support the latest version of ipython.
# Thus, the preview does not work in Colab. However, whenever possible we
# strongly recommend to use the scene preview mode.
try: # detect if the notebook runs in Colab
    import google.colab
    colab_compat = True # deactivate preview
except:
    colab_compat = False
resolution = [480,320] # increase for higher quality of renderings

# Allows to exit cell execution in Jupyter
class ExitCell(Exception):
    def _render_traceback_(self):
        pass

# Import Sionna
try:
    import sionna
except ImportError as e:
    # Install Sionna if package is not already installed
    import os
    os.system("pip install sionna")
    import sionna

# Configure the notebook to use only a single GPU and allocate only as much memory as needed
# For more details, see https://www.tensorflow.org/guide/gpu
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)
# Avoid warnings from TensorFlow
tf.get_logger().setLevel('ERROR')

tf.random.set_seed(1) # Set global random seed for reproducibility


## <center> Importação de bibliotecas necessárias</center>
Como todos os programas em python, bibliotecas são necessárias para correr o programa.
Então iremos importá-las ...

In [ ]:
%matplotlib inline 
import matplotlib.pyplot as plt 
import numpy as np
import time
import math
# Bibliotecas principais para o funcionamento do Sionna
from sionna.rt import load_scene, Transmitter, Receiver, PlanarArray, Camera

# <center>Criação de um cenário(scene)</center>
### Carregar cena local
<br>
Agora que já importamos todas as bibliotecas necessárias para o uso do programa, podemos prosseguir ao uso do programa, começando por carregar a cena.<br>
<br>Conceitos a saber<br>
Usar ../ para retornar à pasta atrás<br>
Sempre que indicar uma pasta seguinte deve usar "/"(Usar esta barra e não a barra "&#92;")<br>
Ao indicar o localização do ficheiro a carregar devemos sempre ter em conta a pasta onde nos encontramos e a partir daí usar /pasta_seguinte ou ../pasta_anterior<br>
<br>
<br>Supondo que o nosso ficheiro se encontra numa pasta chamada Sionna(e esta pasta está no ambiente de trabalho) e que o ficheiro que estamos a usar está na raiz(pasta default do Jupyter)
<br>Devemos usar load_scene("Desktop/Sionna/Nome_atribuido_ao_ficheiro.xml")<br>
<br>Supondo que o nosso ficheiro se encontra em 2 pastas anteriores à atual seguida do Ambiente de Trabalho(Desktop), seguido da pasta Sionna
<br>Devemos usar load_scene("../../Desktop/Sionna/Nome_atribuido_ao_ficheiro.xml")<br>
<br>
<br>

#### Saber Localização do Ficheiro
Antes de aplicar os conceitos para carregar a scene, caso não saibas a localização do ficheiro no PC, podes usar o comando <b>os.getcwd()</b>, ou <b>os.listdir()</b> para verificar os arquivos naquela past ou <b>os.listdir</b>(Atributos anteriores dos caminhos)
<br>Colocando os atributos/conceitos entre parenteses(apenas para orientação[não obrigatório para correr o programa]) 
<br>
<br>Verifica os exemplos abaixo para compreenderes melhor

In [ ]:
print("Localização do projeto:\n");
print(os.getcwd())
print("\nArquivos juntamente com o projeto:\n")
print(os.listdir())

In [ ]:
print("Arquivos fora da pasta do projeto, na pasta anterior e na pasta Blender:\n")
print(os.listdir("../Code/")) #Alterar para pasta à escolha

### <center>Carregar cena(scene)</center>
<br>Agora aplicando os conceitos anteriores: de load_scene(caminho do ficheiro)
<br>E usando scene.preview() para visualizar a cena(scene)

In [ ]:
scene = load_scene("../Blender/Box.xml") #Configurar o caminho pretendido
scene.preview()                          #Visualizar a cena

# <center>Criação de antenas</center>


### <center>Array para Transmissores e Recetores</center>
Primeiramente necessitamos criar os arrays que consiste em Paineis(do[s] Transmissor[es] e Recetor[es])

In [ ]:
# Configura a array de antenas para o transmissor
scene.tx_array = PlanarArray(
                          num_rows=8,             # Linhas
                          num_cols=2,             # Colunas
                          vertical_spacing=0.7,   # Espaço Vertical entre antenas
                          horizontal_spacing=0.5, # Espaço Horizontal entre antenas
                          pattern="iso",          # “iso”(Isotropic antenna), “dipole”(Short dipole), “hw_dipole”(Half-wavelength dipole), “tr38901”(3GPP TR 38.901)
                          polarization="VH")      # V(Vertical/single polarization), H(Horinzontal/single polarization) ou VH/Cross(dual polarization)

# Configura a array de antenas para todos os recetores
scene.rx_array = PlanarArray(num_rows=1,             # Linhas
                             num_cols=1,             # Colunas
                             vertical_spacing=0.5,   # Espaço Vertical entre antenas
                             horizontal_spacing=0.5, # Espaço Horizontal entre antenas
                             pattern="iso",          # “iso”(Isotropic antenna), “dipole”(Short dipole), “hw_dipole”(Half-wavelength dipole), “tr38901”(3GPP TR 38.901) --> https://nvlabs.github.io/sionna/_modules/sionna/rt/antenna.html#iso_pattern
                             polarization="cross")      # V(Vertical/single polarization), H(Horinzontal/single polarization) ou VH/Cross(dual polarization)



### <center>Antenas para Transmissores e Recetores</center>
Seguidamente, necessitamos criar as próprias antenas(do Transmissor e Recetor)

In [ ]:
# Cria um transmissor
tx = Transmitter(name="tx",
              position=[14.77,31.4,20.76],
              orientation=[0,0,0])



rx0 = Receiver(name="rx0",
           position=[5,10,9],
           orientation=[0,0,0])
rx1 = Receiver(name="rx1",
           position=[5,15,10],
           orientation=[0,0,0])
rx2 = Receiver(name="rx2",
           position=[9,15,50],
           orientation=[0,0,0]) 


### <center>Adicionar à cena</center>
Seguidamente, adicioná-los ao nosso projeto(cena[scene])
<br>E posicionar a camera(do preview, pelo seguimento do sinal, do transmissor para o recetor) --> tx.lookat()

In [ ]:
#Adiciona o transmissor à cena
scene.add(tx)
# Adiciona os recetores à cena
scene.add(rx0)
scene.add(rx1)  # Adiciona o novo recetor à cena
scene.add(rx2)
# TX aponta para o primeiro RX
tx.look_at(rx0)

# <center>Criação dos caminhos do sinal</center><p><center>Paths</center>
Para proceder à criação dos caminhos percorridos pelo sinal devemos usar a função compute_paths() juntamente com a scene ---------------> <b>scene.compute_paths()</b>
<p>Depois para visualizarmos como ficou, usamos a função preview, indicando os paths(para os visualizar) -----------------------------------------> <b>scene.preview(paths=paths)</b>

In [ ]:
paths = scene.compute_paths() 
scene.preview(paths=paths) 
scene.render_to_file(camera="scene-cam-0", filename="scene.png",
                         resolution=[650,500])

# <center>Detalhes dos caminhos</center>
Agora iremos verificar as propriedades/detalhes para cada caminho criado e armazená-las num ficheiro criado com o nome na variavel <b>file_created</b>

In [ ]:
file_created = 'pathsproperties.txt'
with open(file_created, 'w') as file:       #Abertura do documento de texto para escrita de dados
    for path_idx in range(13):              #Enquanto a variavel path_ids(quantidade de caminhos)< máximo(13)...

                        #Imprimir valores na "Porta Série/Terminal"
        print(f"\n--- Resultado para o caminho {path_idx} ---")
        print(f"Coeficiente de Canal de Propagação(channel_coefficient): {paths.a[0,0,0,0,0,path_idx, 0].numpy()}")
        print(f"Atraso de propagação(Propagation Delay): {paths.tau[0,0,0,path_idx].numpy()*1e6:.5f} us")
        #Ângulos de Zenith
        print(f"Ângulo Zenith de Partida (Zenith Angle of Departure): {paths.theta_t[0,0,0,path_idx]:.4f} rad")
        print(f"Ângulo Zenith de Chegada (Zenith Angle of Arrival): {paths.theta_r[0,0,0,path_idx]:.4f} rad")
        #Ângulos de Azimuth
        print(f"Ângulo Azimuth de Partida(Azimuth Angle of Departure): {paths.phi_t[0,0,0,path_idx]:.4f} rad")
        print(f"Ângulo Azimuth de Chegada(Azimuth Angle of Arrival): {paths.phi_r[0,0,0,path_idx]:.4f} rad")

                        #Escrita de valores no ficheiro criado(acima)
        file.write(f"\n--- Resultado para o caminho {path_idx} ---\n")
        file.write(f"Coeficiente de Canal de Propagação(channel_coefficient): {paths.a[0,0,0,0,0,path_idx, 0].numpy()}\n")
        file.write(f"Atraso de propagação(Propagation Delay):: {paths.tau[0,0,0,path_idx].numpy()*1e6:.5f} us\n")
        #Ângulos de Zenith
        file.write(f"Ângulo Zenith de Partida (Zenith Angle of Departure): {paths.theta_t[0,0,0,path_idx]:.4f} rad\n")
        file.write(f"Ângulo Zenith de Chegada (Zenith Angle of Arrival): {paths.theta_r[0,0,0,path_idx]:.4f} rad\n")
        #Ângulos de Azimuth
        file.write(f"Ângulo Azimuth de Partida(Azimuth Angle of Departure): {paths.phi_t[0,0,0,path_idx]:.4f} rad\n")
        file.write(f"Ângulo Azimuth de Chegada(Azimuth Angle of Arrival): {paths.phi_r[0,0,0,path_idx]:.4f} rad\n")
print("\n\nResultados guardados em "+file_created) #Informar que foram escritos dos os resultados no ficheiro definido em file_created

## <center>Coverage Map</center>
Após isso, iremos criar o coverage map para verificar a zona de cobertura do sinal na zona

In [ ]:
# Compute coverage map
cm = scene.coverage_map(max_depth=5,
                        diffraction=True, # Desativar visualização de efeitos na difraçãoDisable to see the effects of diffraction
                        cm_cell_size=(5., 5.), # Grid size das celulas do coverage map no mapa
                        combining_vec=None,
                        precoding_vec=None,
                        num_samples=int(5e6)) # Reduza se o hardware caso não haja memória suficiente

#Impressão do Coverage Map
cm.show()  

### <center>Calculo da coordenada Z(Potência de sinal[coverage_map_dB])</center>
O código abaixo permite retirar o valor da coordenada Z(coverage_map_dB) 
Variando o cmx(Coordenada eixo X) e cmy(Coordenada eixo Y)

In [ ]:
#Definir coordenadas
cmx = 20 #Coordenada x à escolha(que possa ser visualizada no coverage map)
cmy = 4 #Coordenada y à escolha(que possa ser visualizada no coverage map)


#Calculo de nivel através das coordenadas
if cmx > cm.num_cells_x - 1 or cmy > cm.num_cells_y - 1:
    print(f"\033[31mO teu cmx deve ser entre 0 e {cm.num_cells_x - 1} e o teu cmy deve ser entre 0 e {cm.num_cells_y - 1}\033[0m")
elif (cm[0,cmy,cmx].numpy()) != 0:
    coverage_map_watts = cm[0,cmy,cmx].numpy()            #Cálculo da potência de sinal [em Watts]
    coverage_map_dB = 10 * math.log10(coverage_map_watts) #Conversão da potência de sinal[para dB]
    print(f"{coverage_map_dB:.3f} dB")  # Likely prints the same value repeatedly
else:
    coverage_map_dB = 0
    print(f"{coverage_map_dB:.3f} dB")  # Likely prints the same value repeatedly




### <center>Cálculo potência de sinal(coverage_map_dB) para todas as coordenadas </center>
x[Cell index (X-assis)], y[Cell index (Y-assis)], 
De modo a obter z(coverage_map_dB) para todos os pontos possíveis, obtendo assim toda a cobertura de sinal

In [ ]:
#Definição de niveis máximos das coordenadas x,y do coverage map
cmx_max = cm.num_cells_x-1
cmy_max = cm.num_cells_y-1

#Para todos os ponto x,y vai ser calculado o z 
def export(x, y, z):
  with open("variaveis.txt", "a") as arquivo:
    arquivo.write(f"{x};{y};{z}\n")  # Add newline for proper formatting



for x in range(cmx_max):
  for y in range(cmy_max):
    z_watts = cm[0, y, x].numpy()  # Access element using correct indexing

    if z_watts:
        z = 10 * math.log10(z_watts)
        print(x, y, z)
        export(x, y, z)
    else:
        z = 0  # Set z to a meaningful value
print("\n\nCoordenadas armazenadas em \"variaveis.txt\"")